In [ ]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

GOLD = "lh_gold.dbo"


def read_table(lakehouse: str, table: str):
    for name in (f"{lakehouse}.dbo.{table}", f"{lakehouse}.{table}"):
        try:
            return spark.table(name)
        except Exception:
            continue
    raise ValueError(f"Could not read {table} from {lakehouse}")


def save_dim(df, table_name: str):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.{table_name}")


def non_empty(col):
    trimmed = F.trim(col)
    return F.when(trimmed != "", trimmed)


silver = read_table("lh_silver", "silver_flights")
lkp_airline = read_table("lh_bronze", "bronze_lkp_airline_id")
lkp_airport = read_table("lh_bronze", "bronze_lkp_airport_id")

# ---------- dim_date ----------
bounds = silver.agg(F.min("flight_date").alias("min_d"), F.max("flight_date").alias("max_d")).first()
calendar_start = date(bounds["min_d"].year, 1, 1)
calendar_end = date(bounds["max_d"].year, 12, 31)
d = F.col("date")

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{calendar_start}'), to_date('{calendar_end}'), interval 1 day)) AS date")
    .select(
        F.date_format(d, "yyyyMMdd").cast("int").alias("date_key"),
        d.alias("date"),
        F.year(d).alias("year"),
        F.quarter(d).alias("quarter"),
        F.concat(F.lit("Q"), F.quarter(d)).alias("quarter_label"),
        F.month(d).alias("month"),
        F.date_format(d, "MMMM").alias("month_name"),
        F.date_format(d, "MMM").alias("month_short"),
        F.date_format(d, "yyyy-MM").alias("year_month"),
        F.date_format(d, "MMM yyyy").alias("month_year_label"),
        (F.year(d) * 100 + F.month(d)).alias("year_month_sort"),
        F.trunc(d, "month").alias("month_start_date"),
        F.dayofmonth(d).alias("day_of_month"),
        (((F.dayofweek(d) + 5) % 7) + 1).alias("day_of_week"),
        F.date_format(d, "EEEE").alias("day_name"),
        F.date_format(d, "EEE").alias("day_short"),
        ((((F.dayofweek(d) + 5) % 7) + 1) >= 6).alias("is_weekend"),
        F.when(d < F.to_date(F.lit("2020-03-01")), F.lit("Pre-COVID"))
         .when(d < F.to_date(F.lit("2022-01-01")), F.lit("Disruption"))
         .when(d < F.to_date(F.lit("2024-01-01")), F.lit("Recovery"))
         .otherwise(F.lit("Post-COVID")).alias("covid_period"),
        F.when(d < F.to_date(F.lit("2020-03-01")), F.lit(1))
         .when(d < F.to_date(F.lit("2022-01-01")), F.lit(2))
         .when(d < F.to_date(F.lit("2024-01-01")), F.lit(3))
         .otherwise(F.lit(4)).alias("covid_period_sort"),
        d.between(F.lit(bounds["min_d"]), F.lit(bounds["max_d"])).alias("has_flight_data"),
    )
)
save_dim(dim_date, "dim_date")

# ---------- dim_airline ----------
airline_usage = (
    silver.groupBy("airline_dot_id", "airline_code")
    .agg(F.min("flight_date").alias("first_flight_date"), F.max("flight_date").alias("last_flight_date"))
)
latest_airline_code = (
    airline_usage
    .withColumn("rn", F.row_number().over(Window.partitionBy("airline_dot_id").orderBy(F.desc("last_flight_date"))))
    .filter(F.col("rn") == 1)
    .select("airline_dot_id", "airline_code")
)
airline_span = airline_usage.groupBy("airline_dot_id").agg(
    F.min("first_flight_date").alias("first_flight_date"),
    F.max("last_flight_date").alias("last_flight_date"),
)
airline_names = lkp_airline.select(
    F.trim(F.col("code")).cast("int").alias("airline_dot_id"),
    non_empty(F.regexp_extract(F.col("description"), r"^(.*):\s*[^:]+$", 1)).alias("lookup_name"),
)

dim_airline = (
    latest_airline_code
    .join(airline_span, "airline_dot_id")
    .join(airline_names, "airline_dot_id", "left")
    .select(
        F.col("airline_dot_id").alias("airline_key"),
        "airline_code",
        F.coalesce(F.col("lookup_name"), F.col("airline_code")).alias("airline_name"),
        F.concat(F.coalesce(F.col("lookup_name"), F.col("airline_code")), F.lit(" ("), F.col("airline_code"), F.lit(")")).alias("airline_label"),
        F.col("lookup_name").isNull().alias("is_name_from_code"),
        "first_flight_date",
        "last_flight_date",
    )
)
save_dim(dim_airline, "dim_airline")

# ---------- dim_airport ----------
def airport_side(prefix: str):
    return silver.select(
        F.col(f"{prefix}_airport_id").alias("airport_id"),
        F.col(f"{prefix}_airport_code").alias("airport_code"),
        F.col(f"{prefix}_city_name").alias("city_name"),
        F.col(f"{prefix}_state_code").alias("state_code"),
        "flight_date",
    )

airport_usage = (
    airport_side("origin").unionByName(airport_side("dest"))
    .groupBy("airport_id", "airport_code", "city_name", "state_code")
    .agg(F.min("flight_date").alias("first_flight_date"), F.max("flight_date").alias("last_flight_date"))
)
latest_airport_attrs = (
    airport_usage
    .withColumn("rn", F.row_number().over(Window.partitionBy("airport_id").orderBy(F.desc("last_flight_date"))))
    .filter(F.col("rn") == 1)
    .select("airport_id", "airport_code", "city_name", "state_code")
)
airport_span = airport_usage.groupBy("airport_id").agg(
    F.min("first_flight_date").alias("first_flight_date"),
    F.max("last_flight_date").alias("last_flight_date"),
)
airport_names = lkp_airport.select(
    F.trim(F.col("code")).cast("int").alias("airport_id"),
    non_empty(F.regexp_extract(F.col("description"), r"^[^:]*:\s*(.*)$", 1)).alias("lookup_name"),
)

dim_airport = (
    latest_airport_attrs
    .join(airport_span, "airport_id")
    .join(airport_names, "airport_id", "left")
    .select(
        F.col("airport_id").alias("airport_key"),
        "airport_code",
        F.coalesce(F.col("lookup_name"), F.col("airport_code")).alias("airport_name"),
        F.coalesce(non_empty(F.regexp_extract(F.col("city_name"), r"^(.*),[^,]*$", 1)), F.col("city_name")).alias("city"),
        "state_code",
        F.col("city_name").alias("city_state"),
        F.concat(F.col("airport_code"), F.lit(" - "), F.col("city_name")).alias("airport_label"),
        F.col("lookup_name").isNull().alias("is_name_from_code"),
        "first_flight_date",
        "last_flight_date",
    )
)
save_dim(dim_airport, "dim_airport")

# ---------- Summary ----------
dd = spark.table(f"{GOLD}.dim_date")
da = spark.table(f"{GOLD}.dim_airline")
dp = spark.table(f"{GOLD}.dim_airport")

print("=== dim_date ===")
print(f"Rows: {dd.count():,} | {calendar_start} to {calendar_end} | days with flight data: {dd.filter('has_flight_data').count():,}")
for r in dd.groupBy("covid_period_sort", "covid_period").agg(F.min("date").alias("start"), F.max("date").alias("end")).orderBy("covid_period_sort").collect():
    print(f"  {r['covid_period']}: {r['start']} to {r['end']}")

print("\n=== dim_airline ===")
print(f"Rows: {da.count()} | names from code fallback: {da.filter('is_name_from_code').count()}")
for r in da.orderBy("airline_code").collect():
    print(f"  {r['airline_code']} | {r['airline_name']} | {r['first_flight_date']} to {r['last_flight_date']}")

print("\n=== dim_airport ===")
print(f"Rows: {dp.count():,} | names from code fallback: {dp.filter('is_name_from_code').count()}")
for r in dp.filter(F.col("airport_code").isin("ATL", "ORD", "JFK")).orderBy("airport_code").collect():
    print(f"  {r['airport_key']} | {r['airport_label']} | {r['airport_name']} | city '{r['city']}', state {r['state_code']}")

=== dim_date ===
Rows: 2,922 | 2019-01-01 to 2026-12-31 | days with flight data: 2,738
  Pre-COVID: 2019-01-01 to 2020-02-29
  Disruption: 2020-03-01 to 2021-12-31
  Recovery: 2022-01-01 to 2023-12-31
  Post-COVID: 2024-01-01 to 2026-12-31

=== dim_airline ===
Rows: 18 | names from code fallback: 0
  9E | Endeavor Air Inc. | 2019-01-01 to 2024-12-31
  AA | American Airlines Inc. | 2019-01-01 to 2026-06-30
  AS | Alaska Airlines Inc. | 2019-01-01 to 2026-06-30
  B6 | JetBlue Airways | 2019-01-01 to 2026-06-30
  DL | Delta Air Lines Inc. | 2019-01-01 to 2026-06-30
  EV | ExpressJet Airlines LLC d/b/a aha! | 2019-01-01 to 2020-09-30
  F9 | Frontier Airlines Inc. | 2019-01-01 to 2026-06-30
  G4 | Allegiant Air | 2019-01-01 to 2026-06-30
  HA | Hawaiian Airlines Inc. | 2019-01-01 to 2025-12-31
  MQ | Envoy Air | 2019-01-01 to 2026-06-30
  NK | Spirit Air Lines | 2019-01-01 to 2026-05-01
  OH | PSA Airlines Inc. | 2019-01-01 to 2026-06-30
  OO | SkyWest Airlines Inc. | 2019-01-01 to 2026-06-

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

GOLD = "lh_gold.dbo"
FACT_TABLE = f"{GOLD}.fact_flights"
DIM_AIRLINE = f"{GOLD}.dim_airline"


def read_table(lakehouse: str, table: str):
    for name in (f"{lakehouse}.dbo.{table}", f"{lakehouse}.{table}"):
        try:
            return spark.table(name)
        except Exception:
            continue
    raise ValueError(f"Could not read {table} from {lakehouse}")


def table_exists(name: str) -> bool:
    try:
        spark.table(name).limit(0).collect()
        return True
    except Exception:
        return False


silver = read_table("lh_silver", "silver_flights")

# ---------- Like-for-like flag on dim_airline ----------
total_periods = silver.select("period_key").distinct().count()
airline_months = silver.groupBy(F.col("airline_dot_id").alias("airline_key")).agg(
    F.countDistinct("period_key").alias("months_reported")
)
enriched = (
    spark.table(DIM_AIRLINE)
    .drop("months_reported", "reports_full_period")
    .join(airline_months, "airline_key", "left")
    .withColumn("reports_full_period", F.col("months_reported") == F.lit(total_periods))
)
enriched_rows = enriched.collect()
spark.createDataFrame(enriched_rows, enriched.schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_AIRLINE)

full_period = [r["airline_code"] for r in enriched_rows if r["reports_full_period"]]
print(f"Airlines reporting all {total_periods} months: {len(full_period)} {sorted(full_period)}")

# ---------- Months to process ----------
silver_periods = silver.groupBy("period_key").agg(F.max("_silver_processed_at").alias("silver_processed_at"))
fact_exists = table_exists(FACT_TABLE)

if fact_exists:
    fact_periods = spark.table(FACT_TABLE).groupBy("period_key").agg(F.max("_silver_processed_at").alias("fact_processed_at"))
    periods_to_process = sorted(
        r["period_key"]
        for r in (
            silver_periods.join(fact_periods, "period_key", "left")
            .filter(F.col("fact_processed_at").isNull() | (F.col("silver_processed_at") > F.col("fact_processed_at")))
            .select("period_key")
            .collect()
        )
    )
else:
    periods_to_process = sorted(r["period_key"] for r in silver_periods.select("period_key").collect())

print(f"Months to process: {len(periods_to_process)}")

# ---------- Build fact ----------
if not periods_to_process:
    print("fact_flights is already up to date. Nothing to do.")
else:
    def flag(col_name):
        return F.col(col_name).cast("int")

    fact = (
        silver.filter(F.col("period_key").isin(periods_to_process))
        .select(
            "period_key",
            F.date_format("flight_date", "yyyyMMdd").cast("int").alias("date_key"),
            F.col("airline_dot_id").alias("airline_key"),
            F.col("origin_airport_id").alias("origin_airport_key"),
            F.col("dest_airport_id").alias("dest_airport_key"),
            "flight_number",
            "tail_number",

            "flight_status",
            "cancellation_reason",
            "scheduled_departure_hour",
            "scheduled_arrival_hour",

            F.lit(1).alias("scheduled_flag"),
            (F.col("flight_status") == "Operated").cast("int").alias("operated_flag"),
            flag("is_cancelled").alias("cancelled_flag"),
            flag("is_diverted").alias("diverted_flag"),
            flag("is_delay_metric_eligible").alias("delay_eligible_flag"),
            flag("is_on_time_arrival").alias("on_time_arrival_flag"),
            flag("is_arrival_delayed_15").alias("arrival_delayed_15_flag"),
            flag("is_departure_delayed_15").alias("departure_delayed_15_flag"),

            "departure_delay_min",
            "departure_delay_nonneg_min",
            "arrival_delay_min",
            "arrival_delay_nonneg_min",
            "taxi_out_min",
            "taxi_in_min",
            "scheduled_elapsed_min",
            "actual_elapsed_min",
            "air_time_min",
            "distance_miles",

            "carrier_delay_min",
            "weather_delay_min",
            "nas_delay_min",
            "security_delay_min",
            "late_aircraft_delay_min",

            flag("dq_extreme_delay").alias("dq_extreme_delay_flag"),
            flag("dq_security_code_2020").alias("dq_security_code_2020_flag"),
            flag("dq_missing_arrival_data").alias("dq_missing_arrival_data_flag"),

            "_silver_processed_at",
            F.current_timestamp().alias("_gold_processed_at"),
        )
    )

    writer = fact.write.format("delta").partitionBy("period_key")
    if fact_exists:
        writer.mode("overwrite").option("partitionOverwriteMode", "dynamic").saveAsTable(FACT_TABLE)
    else:
        writer.mode("overwrite").saveAsTable(FACT_TABLE)

    result = spark.table(FACT_TABLE)
    print(f"{FACT_TABLE}: {result.count():,} rows | {len(result.columns)} columns | {result.select('period_key').distinct().count()} months")

Airlines reporting all 90 months: 12 ['AA', 'AS', 'B6', 'DL', 'F9', 'G4', 'MQ', 'OH', 'OO', 'UA', 'WN', 'YX']
Months to process: 90
fact_flights: 49,251,865 rows | 39 columns | 90 months


In [ ]:
from pyspark.sql import functions as F

GOLD = "lh_gold.dbo"


def read_table(lakehouse: str, table: str):
    for name in (f"{lakehouse}.dbo.{table}", f"{lakehouse}.{table}"):
        try:
            return spark.table(name)
        except Exception:
            continue
    raise ValueError(f"Could not read {table} from {lakehouse}")


def cnt(condition):
    return F.sum(F.when(condition, 1).otherwise(0))


fact = spark.table(f"{GOLD}.fact_flights")
dim_date = spark.table(f"{GOLD}.dim_date")
dim_airline = spark.table(f"{GOLD}.dim_airline")
dim_airport = spark.table(f"{GOLD}.dim_airport")
silver = read_table("lh_silver", "silver_flights")

failures = []


def check(name, passed, detail=""):
    print(f"{'PASS' if passed else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    if not passed:
        failures.append(name)


print("=== Strict tests ===")

# 1. Row reconciliation
recon = (
    silver.groupBy("period_key").agg(F.count("*").alias("silver_rows"))
    .join(fact.groupBy("period_key").agg(F.count("*").alias("fact_rows")), "period_key", "full_outer")
    .filter(F.coalesce(F.col("silver_rows"), F.lit(0)) != F.coalesce(F.col("fact_rows"), F.lit(0)))
    .collect()
)
check(
    "1. Fact rows match Silver for every month",
    not recon,
    f"mismatches: {[(r['period_key'], r['silver_rows'], r['fact_rows']) for r in recon]}" if recon else "",
)

# 2. Dimension keys unique and not null
dim_issues = {}
for name, df, key in [("dim_date", dim_date, "date_key"), ("dim_airline", dim_airline, "airline_key"), ("dim_airport", dim_airport, "airport_key")]:
    duplicates = df.groupBy(key).count().filter(F.col("count") > 1).count()
    nulls = df.filter(F.col(key).isNull()).count()
    if duplicates or nulls:
        dim_issues[name] = {"duplicates": duplicates, "nulls": nulls}
check("2. Dimension keys unique and not null", not dim_issues, f"issues: {dim_issues}" if dim_issues else "")

# 3. Referential integrity
fk_map = [
    ("date_key", dim_date, "date_key"),
    ("airline_key", dim_airline, "airline_key"),
    ("origin_airport_key", dim_airport, "airport_key"),
    ("dest_airport_key", dim_airport, "airport_key"),
]
null_fks = fact.agg(*[cnt(F.col(fk).isNull()).alias(fk) for fk, _, _ in fk_map]).first().asDict()
orphans = {
    fk: fact.select(fk).distinct()
        .join(F.broadcast(dim.select(F.col(pk).alias(fk))), fk, "left_anti")
        .count()
    for fk, dim, pk in fk_map
}
ri_issues = {fk: {"null_keys": null_fks[fk], "orphan_keys": orphans[fk]} for fk, _, _ in fk_map if null_fks[fk] or orphans[fk]}
check("3. Every fact key matches a dimension row", not ri_issues, f"issues: {ri_issues}" if ri_issues else "")

# 4. Measure reconciliation
fm = fact.agg(
    F.sum("operated_flag").alias("operated"),
    F.sum("cancelled_flag").alias("cancelled"),
    F.sum("diverted_flag").alias("diverted"),
    F.sum("delay_eligible_flag").alias("eligible"),
    F.sum("on_time_arrival_flag").alias("on_time"),
    F.sum("arrival_delayed_15_flag").alias("delayed_15"),
    F.sum("arrival_delay_nonneg_min").alias("arrival_delay_minutes"),
).first().asDict()

sm = silver.agg(
    cnt(F.col("flight_status") == "Operated").alias("operated"),
    cnt(F.col("is_cancelled")).alias("cancelled"),
    cnt(F.col("is_diverted")).alias("diverted"),
    cnt(F.col("is_delay_metric_eligible")).alias("eligible"),
    cnt(F.col("is_on_time_arrival")).alias("on_time"),
    cnt(F.col("is_arrival_delayed_15")).alias("delayed_15"),
    F.sum("arrival_delay_nonneg_min").alias("arrival_delay_minutes"),
).first().asDict()

measure_issues = {k: (sm[k], fm[k]) for k in sm if sm[k] != fm[k]}
if fm["on_time"] + fm["delayed_15"] != fm["eligible"]:
    measure_issues["on_time_plus_delayed_vs_eligible"] = (fm["on_time"] + fm["delayed_15"], fm["eligible"])
check(
    "4. Measures reconcile to Silver",
    not measure_issues,
    f"(silver, fact) mismatches: {measure_issues}" if measure_issues else
    f"operated {fm['operated']:,} | on-time {fm['on_time']:,} | delayed {fm['delayed_15']:,} | eligible {fm['eligible']:,}",
)

# ---------- First insights ----------
print("\n=== First insights by COVID period ===")
period_rows = (
    fact
    .join(F.broadcast(dim_date.select("date_key", "covid_period", "covid_period_sort")), "date_key")
    .join(F.broadcast(dim_airline.select("airline_key", "reports_full_period")), "airline_key")
    .groupBy("covid_period_sort", "covid_period")
    .agg(
        F.countDistinct("period_key").alias("months"),
        F.sum("scheduled_flag").alias("scheduled"),
        F.sum("operated_flag").alias("operated"),
        F.sum("cancelled_flag").alias("cancelled"),
        F.sum("on_time_arrival_flag").alias("on_time"),
        F.sum("delay_eligible_flag").alias("eligible"),
        F.sum(F.when(F.col("reports_full_period"), F.col("operated_flag"))).alias("operated_lfl"),
        F.sum(F.when(F.col("reports_full_period"), F.col("on_time_arrival_flag"))).alias("on_time_lfl"),
        F.sum(F.when(F.col("reports_full_period"), F.col("delay_eligible_flag"))).alias("eligible_lfl"),
    )
    .orderBy("covid_period_sort")
    .collect()
)

for r in period_rows:
    print(
        f"{r['covid_period']} ({r['months']} months)\n"
        f"  Operated flights/month: all airlines {r['operated'] / r['months']:,.0f} | like-for-like {r['operated_lfl'] / r['months']:,.0f}\n"
        f"  On-time arrival: all airlines {r['on_time'] / r['eligible']:.1%} | like-for-like {r['on_time_lfl'] / r['eligible_lfl']:.1%}\n"
        f"  Cancellation rate: {r['cancelled'] / r['scheduled']:.2%}"
    )

print(f"\nResult: {'ALL STRICT TESTS PASSED' if not failures else f'{len(failures)} TEST(S) FAILED'}")
if failures:
    raise AssertionError(f"Gold quality gate failed: {failures}")

=== Strict tests ===
PASS | 1. Fact rows match Silver for every month
PASS | 2. Dimension keys unique and not null
PASS | 3. Every fact key matches a dimension row
PASS | 4. Measures reconcile to Silver | operated 48,068,798 | on-time 38,736,464 | delayed 9,332,329 | eligible 48,068,793

=== First insights by COVID period ===
Pre-COVID (14 months)
  Operated flights/month: all airlines 602,541 | like-for-like 528,687
  On-time arrival: all airlines 81.6% | like-for-like 81.6%
  Cancellation rate: 1.71%
Disruption (22 months)
  Operated flights/month: all airlines 414,111 | like-for-like 360,637
  On-time arrival: all airlines 86.0% | like-for-like 85.8%
  Cancellation rate: 3.92%
Recovery (24 months)
  Operated flights/month: all airlines 553,142 | like-for-like 500,861
  On-time arrival: all airlines 79.2% | like-for-like 79.2%
  Cancellation rate: 1.98%
Post-COVID (30 months)
  Operated flights/month: all airlines 574,912 | like-for-like 546,830
  On-time arrival: all airlines 78.3% 